# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow - Post processing

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
//#r "BoSSSpad/BoSSSpad.dll"
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
using BoSSS.Solution.NSECommon;

In [53]:
//var myBatch = ExecutionQueues[1];
var myBatch = GetDefaultQueue();
myBatch

DeploymentBaseDirectory,D:\local\binaries
DeployRuntime,False
RuntimeLocation,<null>
Name,LocalPC
DotnetRuntime,dotnet
BatchInstructionDir,<null>
AllowedDatabasesPaths,[ D:\local\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow_ConvStudy", myBatch);

In [5]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p3_J1310720	4/16/2025 3:00:38 PM	c773aee2...
#1: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_p3_J1310720	4/16/2025 1:59:46 PM	bf803105...
#2: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Pressure_Outlet_ManSol_p3_J1310720	4/16/2025 2:05:45 PM	22ff0bb8...
#3: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J786432	4/16/2025 2:57:51 PM	20de008b...
#4: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p3_J327680	4/16/2025 2:59:38 PM	45a52fbb...
#5: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J196608	4/16/2025 2:57:07 PM	59d02e3f...
#6: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p3_J81920	4/16/2025 2:59:10 PM	3d9d15e3...
#7: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J49152	4/16/2025 2:56:52 PM	66264cf1...
#8: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p3_

In [6]:
// var delSess = sessions.Where(sess => sess.Name.Contains($"KovasznayFlow_ConvStudy_Dong_OutFlow"));
// delSess

In [7]:
// foreach (var sess in delSess) {
//     sess.Delete(true);
// }

## Studies collection

In [8]:
string[] studies = { "Pressure_Outlet", "Dong_OutFlow", "Pressure_Outlet_ManSol", "Dong_OutFlow_ManSol" };

In [9]:
int[] polyDegs = { 2, 3 };

In [ ]:
List<(string Name, ISessionInfo[] Sessions)> studySessions = new List<(string, ISessionInfo[])>();

foreach (string study in studies) {
    foreach (int polyD in polyDegs) {
        var studySess = sessions.Where(sess => sess.Name.Contains($"KovasznayFlow_ConvStudy")
                                            && sess.Name.Contains("_" + study + "_p")
                                            && sess.Name.Contains($"_p{polyD}_")
                                            && sess.SuccessfulTermination());

        var studySess_sort = studySess.OrderBy(sess => sess.Timesteps.Last().Grid.NumberOfCells);

        string studyName = study + $"-p{polyD}";
        studySessions.Add((studyName, studySess_sort.Skip(1).ToArray()));
    }
}

studySessions

index,value
0,"(Pressure_Outlet-p2, BoSSS.Foundation.IO.ISessionInfo[])"
1,"(Pressure_Outlet-p3, BoSSS.Foundation.IO.ISessionInfo[])"
2,"(Dong_OutFlow-p2, BoSSS.Foundation.IO.ISessionInfo[])"
3,"(Dong_OutFlow-p3, BoSSS.Foundation.IO.ISessionInfo[])"
4,"(Pressure_Outlet_ManSol-p2, BoSSS.Foundation.IO.ISessionInfo[])"
5,"(Pressure_Outlet_ManSol-p3, BoSSS.Foundation.IO.ISessionInfo[])"
6,"(Dong_OutFlow_ManSol-p2, BoSSS.Foundation.IO.ISessionInfo[])"
7,"(Dong_OutFlow_ManSol-p3, BoSSS.Foundation.IO.ISessionInfo[])"


In [12]:
int numOfStudySessions = 0;
foreach (var study in studySessions) {
    numOfStudySessions += study.Sessions.Count();
}

numOfStudySessions

64

In [ ]:
//studySessions[6].Sessions

#0: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J48	4/16/2025 2:55:39 PM	b7f12169...
#1: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J192	4/16/2025 2:55:50 PM	f05436ff...
#2: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J768	4/16/2025 2:56:07 PM	6b4e4a9d...
#3: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J3072	4/16/2025 2:56:22 PM	4c4e8284...
#4: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J12288	4/16/2025 2:56:34 PM	c3568c54...
#5: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J49152	4/16/2025 2:56:52 PM	66264cf1...
#6: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J196608	4/16/2025 2:57:07 PM	59d02e3f...
#7: KovasznayFlow_ConvStudy	KovasznayFlow_ConvStudy_Dong_OutFlow_ManSol_p2_J786432	4/16/2025 2:57:51 PM	20de008b...


In [ ]:
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\bkup-2025May08_000000.KovasznayFlow_ConvStudy");

In [16]:
databases

#0: { Session Count = 108; Grid Count = 27; Path = \\dc3\userspace\smuda\hpccluster\KovasznayFlow_ConvStudy }
#1: { Session Count = 64; Grid Count = 16; Path = \\dc3\userspace\smuda\hpccluster\bkup-2025May08_000000.KovasznayFlow_ConvStudy }


In [ ]:
// foreach (var study in studySessions) {
//     foreach (var sess in study.Sessions) {
//         sess.Copy(databases.Pick(1));
//     }
// }

## Studies evaluation

In [17]:
Formula KovasznayFlow_u = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velX = 1.0 - (Math.Exp(lambda * X[0]) * Math.Cos(2.0 * Math.PI * X[1]));" +
    "return velX; } "
);

In [18]:
Formula KovasznayFlow_v = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velY = (lambda/(2.0 * Math.PI)) * (Math.Exp(lambda * X[0]) * Math.Sin(2.0 * Math.PI * X[1]));" +
    "return velY; } "
);

In [19]:
Formula KovasznayFlow_p = new Formula(
    "Pres",
    false,
    "double Pres(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double Pres = 0.5 * (1.0 - Math.Exp(2.0 * lambda * X[0]));" +
    "return Pres; } "
);

In [ ]:
List<(string Name, double[] hSize, MultidimensionalArray L2norms)> studyL2norms = new List<(string, double[], MultidimensionalArray)>();

string[] fields = new string[] {"VelocityX", "VelocityY", "Pressure"};
foreach (var studySess in studySessions) {
    Console.WriteLine($"processing study {studySess.Name} ... ");
    int numGrids = studySess.Sessions.Count();
    double[] hSize = new double[numGrids];
    MultidimensionalArray L2norms = MultidimensionalArray.Create(fields.Length, numGrids);

    for (int i = 0; i < numGrids; i++) {
        Console.WriteLine($" ... Session {studySess.Sessions.Pick(i).Name}");
        var ts = studySess.Sessions.Pick(i).Timesteps.Last();
        hSize[i] = ts.Grid.GetMeshSize();


        L2norms[0, i] = ts.GetField(fields[0]).L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
            KovasznayFlow_u.EvaluateV(nodes, 0.0, results);
        });

        L2norms[1, i] = ts.GetField(fields[1]).L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
            KovasznayFlow_v.EvaluateV(nodes, 0.0, results);
        });

        // L2norms[2, i] = ts.GetField("Pressure").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
        //     KovasznayFlow_p.EvaluateV(nodes, 0.0, results);
        // });
        double[] PrefPoint = new double[] { -0.49, -0.49 };
        double Prefvalue = ts.GetField(fields[2]).ProbeAt(PrefPoint);
        double PsolValue = KovasznayFlow_p.Evaluate(PrefPoint, 0.0);
        double Pdiff = PsolValue - Prefvalue;
        //Console.WriteLine($"PrefValue = {Prefvalue}; PsolValue = {PsolValue}; Pdiff = {Pdiff}");
        var refPressure = ts.GetField(fields[2]).CloneAs();
        refPressure.AccConstant(Pdiff);
        L2norms[2, i] = refPressure.L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
            KovasznayFlow_p.EvaluateV(nodes, 0.0, results);
        });
    }

    studyL2norms.Add((studySess.Name, hSize, L2norms));
}

In [30]:
Plot2Ddata pltVelX = new Plot2Ddata(); 
pltVelX.Title = "L2 norms - VelocityX";
pltVelX.LogX = true;
pltVelX.LogY = true;

Plot2Ddata pltVelY = new Plot2Ddata(); 
pltVelY.Title = "L2 norms - VelocityY";
pltVelY.LogX = true;
pltVelY.LogY = true;

Plot2Ddata pltP = new Plot2Ddata(); 
pltP.Title = "L2 norms - Pressure";
pltP.LogX = true;
pltP.LogY = true;


var gp = new Gnuplot();
gp.SetMultiplot(2,2);

for (int i = 0; i < studyL2norms.Count(); i++) {

    var sNorms = studyL2norms[i];

    var fmt = new PlotFormat();  
    fmt.SetLineColorFromIndex(i);

    // multiplot 0, 0 - VelocityX
    pltVelX.AddDataGroup(sNorms.Name, sNorms.hSize, sNorms.L2norms.ExtractSubArrayShallow(0, -1).To1DArray(), fmt);

    // multiplot 0, 1 - VelocityY
    pltVelY.AddDataGroup(sNorms.Name, sNorms.hSize, sNorms.L2norms.ExtractSubArrayShallow(1, -1).To1DArray(), fmt);

    // multiplot 1, 0 - Pressure
    pltP.AddDataGroup(sNorms.Name, sNorms.hSize, sNorms.L2norms.ExtractSubArrayShallow(2, -1).To1DArray(), fmt);
}

gp.SetSubPlot(0, 0, pltVelX.Title);
pltVelX.ToGnuplot(gp);

gp.SetSubPlot(0, 1, pltVelY.Title);
pltVelY.ToGnuplot(gp);

gp.SetSubPlot(1, 0, pltP.Title);
pltP.ToGnuplot(gp);

gp.PlotSVG(xRes:1500,yRes:1000)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -12 
 
 
 
 
 10 -11 
 
 
 
 
 10 -10 
 
 
 
 
 10 -9 
 
 
 
 
 10 -8 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2 norms - VelocityX 
 
 
 PressureOutlet-p2 
 
 
 Pressure O utlet-p2 
 
 
 
 
 
 PressureOutlet-p3 
 
 
 Pressure O utlet-p3 
 
 
 
 
 
 DongOutFlow-p2 
 
 
 Dong O utFlow-p2 
 
 
 
 
 
 DongOutFlow-p3 
 
 
 Dong O utFlow-p3 
 
 
 
 
 
 PressureOutletManSol-p2 
 
 
 Pressure O utlet M anSol-p2 
 
 
 
 
 
 PressureOutletManSol-p3 
 
 
 Pressure O utlet M anSol-p3 
 
 
 
 
 
 DongOutFlowManSol-p2 
 
 
 Dong O utFlow M anSol-p2 
 
 
 
 
 
 DongOutFlowManSol-p3 
 
 
 Dong O utFlow M anSol-p3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -12 
 
 
 
 
 10 -11 
 
 
 
 
 10 -10 
 
 
 
 
 10 -9 
 
 
 
 
 10 -8 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
	<path stroke='black' d='M1085.1,459.1 L1085.1,454.6 M1085.1,77.1 L1085.1,81.6 M1122.1,459.1 L1122.1,454.6 M1122.1,77.1 L1122.1,81.6
 M1148.4,459.1 L1148.4,454.6 M1148.4,77.1 L1148.4,81.6 M1168.8,459.1 L1168.8,454.6 M1168.8,77.1 L1168.8,81.6
 M1185.5,459.1 L1185.5,454.6 M1185.5,77.1 L1185.5,81.6 M1199.6,459.1 L1199.6,454.6 M1199.6,77.1 L1199.6,81.6
 M1211.8,459.1 L1211.8,454.6 M1211.8,77.1 L1211.8,81.6 M1222.5,459.1 L1222.5,454.6 M1222.5,77.1 L1222.5,81.6
 M1232.2,459.1 L1232.2,450.1 M1232.2,77.1 L1232.2,86.1 '/> 
 10 -1 
 
 
 
	<path stroke='black' d='M1295.5,459.1 L1295.5,454.6 M1295.5,77.1 L1295.5,81.6 M1332.6,459.1 L1332.6,454.6 M1332.6,77.1 L1332.6,81.6
 M1358.9,459.1 L1358.9,454.6 M1358.9,77.1 L1358.9,81.6 M1379.3,459.1 L1379.3,454.6 M1379.3,77.1 L1379.3,81.6
 M1395.9,459.1 L1395.9,454.6 M1395.9,77.1 L1395.9,81.6 M1410.0,459.1 L1410.0,454.6 M1410.0,77.1 L1410.0,81.6
 M1422.2,459.1 L1422.2,454.6 M1422.2,77.1 L1422.2,81.6 M1433.0,459.1 L1433.0,454.6 M1433.0,77.1 L1433.0,81.6
 M1442.6,459.1 L1442.6,450.1 M1442.6,77.1 L1442.6,86.1 '/> 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2 norms - VelocityY 
 
 
 PressureOutlet-p2 
 
 
 Pressure O utlet-p2 
 
 
 
 
 
 PressureOutlet-p3 
 
 
 Pressure O utlet-p3 
 
 
 
 
 
 DongOutFlow-p2 
 
 
 Dong O utFlow-p2 
 
 
 
 
 
 DongOutFlow-p3 
 
 
 Dong O utFlow-p3 
 
 
 
 
 
 PressureOutletManSol-p2 
 
 
 Pressure O utlet M anSol-p2 
 
 
 
 
 
 PressureOutletManSol-p3 
 
 
 Pressure O utlet M anSol-p3 
 
 
 
 
 
 DongOutFlowManSol-p2 
 
 
 Dong O utFlow M anSol-p2 
 
 
 
 
 
 DongOutFlowManSol-p3 
 
 
 Dong O utFlow M anSol-p3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -10 
 
 
 
 
 10 -9 
 
 
 
 
 10 -8 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2 norms - Pressure 
 
 
 PressureOutlet-p2 
 
 
 Pressure O utlet-p2 
 
 
 
 
 
 PressureOutlet-p3 
 
 
 Pressure O utlet-p3 
 
 
 
 
 
 DongOutFlow-p2 
 
 
 Dong O utFlow-p2 
 
 
 
 
 
 DongOutFlow-p3 
 
 
 Dong O utFlow-p3 
 
 
 
 
 
 PressureOutletManSol-p2 
 
 
 Pressure O utlet M anSol-p2 
 
 
 
 
 
 PressureOutletManSol-p3 
 
 
 P

In [26]:
using NUnit.Framework;

In [44]:
void AssertSlopes((string Name, double[] hSize, MultidimensionalArray L2norms) studyNorms, double[] expectedSlopes) {
    for (int d = 0; d < fields.Length; d++) {
        double regSlope = DoubleExtensions.LogLogRegressionSlope(studyNorms.hSize, studyNorms.L2norms.ExtractSubArrayShallow(d, -1).To1DArray());
        Assert.GreaterOrEqual(regSlope, expectedSlopes[d], $"{studyNorms.Name}/{fields[d]}: regression slope below allowed limit");
    }
}

In [45]:
void AsserLowerLimits((string Name, double[] hSize, MultidimensionalArray L2norms) studyNorms, double[] allowedLimits) {
    for (int d = 0; d < fields.Length; d++) {
        double minVal = studyNorms.L2norms.ExtractSubArrayShallow(d, -1).Min();
        Assert.LessOrEqual(minVal, allowedLimits[d], $"{studyNorms.Name}/{fields[d]}: minimum value above allowed limit");
    }
}

In [48]:
foreach (var studNorms in studyL2norms.Where(study => !study.Name.Contains($"_ManSol"))) {
    AssertSlopes(studNorms, new double[] {-0.01, -0.01, -0.01});
    AsserLowerLimits(studNorms, new double[] {1e-2, 1e-2, 1e-2});
}

In [51]:
foreach (var studNorms in studyL2norms.Where(study => study.Name.Contains($"_ManSol-p2"))) {
    AssertSlopes(studNorms, new double[] {3, 3, 2});
    AsserLowerLimits(studNorms, new double[] {1e-8, 1e-8, 1e-6});
}

In [52]:
foreach (var studNorms in studyL2norms.Where(study => study.Name.Contains($"_ManSol-p3"))) {
    AssertSlopes(studNorms, new double[] {4, 4, 3});
    AsserLowerLimits(studNorms, new double[] {1e-11, 1e-11, 1e-9});
}